# Transition Gap Summary

Reads the canonical DTL logs from
`logs/sim2real_transitions/cityflow_{agent}/{network}/baseline_v1_{method}_{setting}_s0/logger/`
(9 tab-separated columns, canonical modes `SIM_TRAIN` / `SIM_TEST` / `REAL_TRAIN` / `REAL_TEST`).

**Selection rules** (per agent / network / setting / method, newest DTL with usable real rows):

- `direct_transfer` — the single zero-shot `REAL_TEST` row.
- `domain_randomization` — top-`TOP_REWARD_K` `REAL_TEST` replay-curve rows by reward, averaged.
- `domain_adaptation` — top-`TOP_REWARD_K` `REAL_TRAIN` rollouts by reward. DA's real curve is logged
  as `REAL_TRAIN` (its 100 posterior-conditioning rollouts are full real evals of the current
  checkpoint); its single `REAL_TEST` row is the terminal *last*-checkpoint eval, and scoring DA on
  that alone would apply a last-weights rule where every other method gets curve-best.
- `gat` / `ugat` / `jlgat` — these runs log no `REAL_TEST`; top-`TOP_REWARD_K` `REAL_TRAIN` rollouts by reward.

**Gap** = selected real travel time − pretrained-tsc CityFlow ATT
(`logs/engine_baselines/{agent}/{network}/cityflow_direct_transfer`, `FINAL_TEST` row). ATT is the headline gap (charts, gap tables); the CSVs also carry queue/delay/throughput gaps
against the same baseline row (`gap_<metric> = real − cityflow`; for throughput negative is worse).
Reward is excluded (engine units). The reference is the pretrained CityFlow eval, **not** the in-run `SIM_TEST`.

`jlgat` needs multiple intersections, so it is absent on the five single-intersection networks
(shows up as 0 in the availability table).

**Note.** Parsing, selection and the CSV export live in `scripts/gap_tables.py` (shared with `make_figures.ipynb`); this notebook is the exploratory companion — availability, pivot tables, bar charts and per-checkpoint traces.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# All parsing + selection logic lives in scripts/gap_tables.py (shared with
# make_figures.ipynb); note the transitions-specific REAL_TRAIN fallback (DA
# logs one terminal REAL_TEST on top of its REAL_TRAIN curve -- the curve is
# scored so every multi-episode method obeys the same top-5 rule). build()
# parses logs/ -> tables/transition_table.csv and returns the raw frame.
from scripts.gap_tables import AGENTS, TRANSITIONS as TASK

results_df = TASK.build(verbose=True)

NETWORK_ORDER = TASK.networks
SETTING_ORDER = TASK.settings
METHOD_ORDER = TASK.methods
SINGLE_SHOT_METHODS = TASK.single_shot
candidate_dtls, real_rows, read_dtl = TASK.candidate_dtls, TASK.real_rows, TASK.read_dtl

# fixed method -> color (Tol bright, CVD-safe); consistent across every figure
METHOD_COLORS = {
    'direct_transfer': '#4477AA',
    'domain_randomization': '#EE6677',
    'domain_adaptation': '#228833',
    'gat': '#CCBB44',
    'ugat': '#66CCEE',
    'jlgat': '#AA3377',
}


## Method Availability

In [4]:
availability = (
    results_df.pivot_table(index=['agent', 'network'], columns='method',
                           values='travel_time', aggfunc='count', observed=False)
    .reindex(columns=METHOD_ORDER)
    .fillna(0).astype(int)
)
availability  # number of settings with a usable run, per method

method                  direct_transfer  domain_randomization  \
agent      network                                              
dqn        tempe_1x1                  4                     4   
           bullhead_1                 4                     4   
           cologne1                   4                     4   
           ingolstadt1                4                     4   
           hz1x1                      4                     4   
           tempe_16                   4                     4   
           bullhead_3                 4                     4   
           cologne3                   4                     4   
           ingolstadt7                4                     4   
           hz4x4                      4                     4   
presslight tempe_1x1                  4                     4   
           bullhead_1                 4                     4   
           cologne1                   4                     4   
           ingolstadt1                4                     4   
           hz1x1                      4                     4   
           tempe_16                   4                     4   
           bullhead_3                 4                     4   
           cologne3                   4                     4   
           ingolstadt7                4                     4   
           hz4x4                      4                     4   

method                  domain_adaptation  gat  ugat  jlgat  
agent      network                                           
dqn        tempe_1x1                    4    4     4      0  
           bullhead_1                   4    4     4      0  
           cologne1                     4    4     4      0  
           ingolstadt1                  4    4     4      0  
           hz1x1                        4    4     4      0  
           tempe_16                     4    4     4      4  
           bullhead_3                   4    4     4      4  
           cologne3                     4    4     4      4  
           ingolstadt7                  4    4     4      4  
           hz4x4                        4    4     4      4  
presslight tempe_1x1                    4    4     4      0  
           bullhead_1                   4    4     4      0  
           cologne1                     4    4     4      0  
           ingolstadt1                  4    4     4      0  
           hz1x1                        4    4     4      0  
           tempe_16                     4    4     4      4  
           bullhead_3                   4    4     4      4  
           cologne3                     4    4     4      4  
           ingolstadt7                  4    4     4      4  
           hz4x4                        4    4     4      4

## Real Travel-Time Tables (selected real rows)

In [5]:
att_tables = {
    agent: (
        results_df[results_df['agent'] == agent]
        .pivot_table(index=['network', 'setting'], columns='method', values='travel_time', observed=False)
        .reindex(columns=METHOD_ORDER)
        .round(1)
    )
    for agent in AGENTS
}
att_tables['dqn']

method                direct_transfer  domain_randomization  \
network     setting                                           
tempe_1x1   setting1            159.3                 158.1   
            setting2            159.3                 158.1   
            setting3            167.0                 164.6   
            setting4           1178.8                 243.8   
bullhead_1  setting1            131.9                 138.6   
            setting2            176.3                 133.7   
            setting3           2147.1                 137.1   
            setting4           2186.9                 151.9   
cologne1    setting1            182.5                 158.0   
            setting2            770.3                 144.7   
            setting3            218.5                 188.6   
            setting4            304.2                 263.0   
ingolstadt1 setting1             97.9                  81.8   
            setting2            102.7                  91.8   
            setting3            121.6                 133.5   
            setting4            275.6                 172.7   
hz1x1       setting1            189.4                 133.9   
            setting2            214.7                 142.1   
            setting3            216.5                 162.4   
            setting4            308.9                 263.8   
tempe_16    setting1             95.6                  93.1   
            setting2             96.4                  93.6   
            setting3            107.3                 101.0   
            setting4            141.7                 131.8   
bullhead_3  setting1            151.3                 140.0   
            setting2            154.8                 143.5   
            setting3            156.4                 149.5   
            setting4            164.9                 156.8   
cologne3    setting1            113.5                 120.6   
            setting2            249.9                 135.3   
            setting3            180.8                 161.8   
            setting4            257.2                 266.3   
ingolstadt7 setting1            111.8                 108.7   
            setting2            121.0                 128.3   
            setting3            140.5                 128.9   
            setting4            305.4                 178.7   
hz4x4       setting1            422.3                 388.5   
            setting2            433.8                 386.1   
            setting3            441.6                 388.2   
            setting4            441.8                 401.1   

method                domain_adaptation    gat   ugat  jlgat  
network     setting                                           
tempe_1x1   setting1              159.1  158.7  159.2    NaN  
            setting2              159.3  158.7  158.6    NaN  
            setting3              167.0  165.1  167.2    NaN  
            setting4              214.3  247.2  238.4    NaN  
bullhead_1  setting1              128.8  130.6  128.5    NaN  
            setting2              134.2  128.2  130.5    NaN  
            setting3              136.1  133.7  131.4    NaN  
            setting4              146.4  189.7  142.8    NaN  
cologne1    setting1              174.9  144.5  143.4    NaN  
            setting2              171.1  128.3  152.2    NaN  
            setting3              202.7  202.0  127.2    NaN  
            setting4              245.5  279.9  206.7    NaN  
ingolstadt1 setting1              105.8  106.0   96.2    NaN  
            setting2               96.4  108.2   88.0    NaN  
            setting3              107.5  131.6  123.2    NaN  
            setting4              161.4  285.4  161.2    NaN  
hz1x1       setting1              134.2  159.2  131.2    NaN  
            setting2              154.2  184.2  155.8    NaN  
            setting3              148.7  186.2  172.5    NaN  
            setting4          

In [6]:
att_tables['presslight']

method                direct_transfer  domain_randomization  \
network     setting                                           
tempe_1x1   setting1            186.8                 158.9   
            setting2            176.5                 160.6   
            setting3            186.0                 169.8   
            setting4            344.9                 289.7   
bullhead_1  setting1            137.3                 138.3   
            setting2            136.9                 172.2   
            setting3            140.7                 144.3   
            setting4            149.8                 148.7   
cologne1    setting1            132.8                 138.8   
            setting2            134.5                 120.3   
            setting3            449.4                 150.8   
            setting4            422.8                 247.2   
ingolstadt1 setting1            103.6                 108.2   
            setting2            111.7                 114.3   
            setting3            142.1                 126.7   
            setting4            221.5                 164.8   
hz1x1       setting1            190.6                 168.6   
            setting2            187.9                 186.7   
            setting3            192.0                 179.1   
            setting4            249.8                 281.8   
tempe_16    setting1            104.7                  98.1   
            setting2            102.6                 109.2   
            setting3            121.7                 107.2   
            setting4            180.3                 143.5   
bullhead_3  setting1            249.1                 271.8   
            setting2            251.8                 172.5   
            setting3            175.2                 255.1   
            setting4           1004.5                 178.4   
cologne3    setting1            114.6                 177.0   
            setting2            186.6                 167.3   
            setting3            175.0                 160.2   
            setting4            240.2                 361.8   
ingolstadt7 setting1            103.1                 109.7   
            setting2            105.4                 115.7   
            setting3            118.3                 133.8   
            setting4            396.9                 199.7   
hz4x4       setting1            407.9                 393.5   
            setting2            407.4                 399.4   
            setting3            411.1                 408.9   
            setting4            437.8                 422.6   

method                domain_adaptation    gat   ugat  jlgat  
network     setting                                           
tempe_1x1   setting1              160.2  161.2  171.5    NaN  
            setting2              163.8  160.9  165.3    NaN  
            setting3              169.0  176.3  168.2    NaN  
            setting4              232.5  267.7  226.6    NaN  
bullhead_1  setting1              137.3  137.2  138.3    NaN  
            setting2              135.7  128.8  133.4    NaN  
            setting3              140.9  136.8  143.2    NaN  
            setting4              149.7  151.9  141.2    NaN  
cologne1    setting1              128.8  160.6  158.8    NaN  
            setting2              154.2  158.4  165.8    NaN  
            setting3              141.0  184.2  200.8    NaN  
            setting4              238.1  268.9  245.8    NaN  
ingolstadt1 setting1              110.3   84.7  101.6    NaN  
            setting2              113.3  102.2  106.5    NaN  
            setting3              134.4  117.2  122.1    NaN  
            setting4              167.6  140.8  140.7    NaN  
hz1x1       setting1              130.2  131.6  136.6    NaN  
            setting2              142.1  152.6  148.4    NaN  
            setting3              187.3  154.2  143.8    NaN  
            setting4          

## Gap vs CityFlow Baseline — `gap = real ATT − pretrained cityflow ATT`

In [7]:
gap_tables = {}
for agent in AGENTS:
    sub = results_df[results_df['agent'] == agent]
    table = (
        sub.pivot_table(index=['network', 'setting'], columns='method', values='gap_vs_cityflow', observed=False)
        .reindex(columns=METHOD_ORDER)
    )
    table.insert(0, 'cityflow_att', sub.groupby(['network', 'setting'], observed=False)['cityflow_att'].first())
    gap_tables[agent] = table.round(1)
gap_tables['dqn']

method                cityflow_att  direct_transfer  domain_randomization  \
network     setting                                                         
tempe_1x1   setting1         143.0             16.3                  15.1   
            setting2         143.0             16.3                  15.1   
            setting3         143.0             24.0                  21.6   
            setting4         143.0           1035.8                 100.8   
bullhead_1  setting1         117.9             14.0                  20.7   
            setting2         117.9             58.4                  15.8   
            setting3         117.9           2029.2                  19.2   
            setting4         117.9           2069.0                  34.0   
cologne1    setting1          42.1            140.4                 115.9   
            setting2          42.1            728.2                 102.6   
            setting3          42.1            176.4                 146.5   
            setting4          42.1            262.1                 220.9   
ingolstadt1 setting1          30.2             67.7                  51.6   
            setting2          30.2             72.5                  61.6   
            setting3          30.2             91.4                 103.3   
            setting4          30.2            245.4                 142.5   
hz1x1       setting1         113.8             75.6                  20.1   
            setting2         113.8            100.9                  28.3   
            setting3         113.8            102.7                  48.6   
            setting4         113.8            195.1                 150.0   
tempe_16    setting1          76.1             19.5                  17.0   
            setting2          76.1             20.3                  17.5   
            setting3          76.1             31.2                  24.9   
            setting4          76.1             65.6                  55.7   
bullhead_3  setting1         125.3             26.0                  14.7   
            setting2         125.3             29.5                  18.2   
            setting3         125.3             31.1                  24.2   
            setting4         125.3             39.6                  31.5   
cologne3    setting1          67.4             46.1                  53.2   
            setting2          67.4            182.5                  67.9   
            setting3          67.4            113.4                  94.4   
            setting4          67.4            189.8                 198.9   
ingolstadt7 setting1          63.7             48.1                  45.0   
            setting2          63.7             57.3                  64.6   
            setting3          63.7             76.8                  65.2   
            setting4          63.7            241.7                 115.0   
hz4x4       setting1         355.1             67.2                  33.4   
            setting2         355.1             78.7                  31.0   
            setting3         355.1             86.5                  33.1   
            setting4         355.1             86.7                  46.0   

method                domain_adaptation    gat   ugat  jlgat  
network     setting                                           
tempe_1x1   setting1               16.1   15.7   16.2    NaN  
            setting2               16.3   15.7   15.6    NaN  
            setting3               24.0   22.1   24.2    NaN  
            setting4               71.3  104.2   95.4    NaN  
bullhead_1  setting1               10.9   12.7   10.6    NaN  
            setting2               16.3   10.3   12.6    NaN  
            setting3               18.2   15.8   13.5    NaN  
            setting4               28.5   71.8   24.9    NaN  
cologne1    setting1              132.8  102.4  101.3    NaN  
            setting2              129.0   86.2  110.1    NaN  
         

In [8]:
gap_tables['presslight']

method                cityflow_att  direct_transfer  domain_randomization  \
network     setting                                                         
tempe_1x1   setting1         143.1             43.7                  15.8   
            setting2         143.1             33.4                  17.5   
            setting3         143.1             42.9                  26.7   
            setting4         143.1            201.8                 146.6   
bullhead_1  setting1         121.0             16.3                  17.3   
            setting2         121.0             15.9                  51.2   
            setting3         121.0             19.7                  23.3   
            setting4         121.0             28.8                  27.7   
cologne1    setting1          44.0             88.8                  94.8   
            setting2          44.0             90.5                  76.3   
            setting3          44.0            405.4                 106.8   
            setting4          44.0            378.8                 203.2   
ingolstadt1 setting1          32.5             71.1                  75.7   
            setting2          32.5             79.2                  81.8   
            setting3          32.5            109.6                  94.2   
            setting4          32.5            189.0                 132.3   
hz1x1       setting1         112.2             78.4                  56.4   
            setting2         112.2             75.7                  74.5   
            setting3         112.2             79.8                  66.9   
            setting4         112.2            137.6                 169.6   
tempe_16    setting1          76.2             28.5                  21.9   
            setting2          76.2             26.4                  33.0   
            setting3          76.2             45.5                  31.0   
            setting4          76.2            104.1                  67.3   
bullhead_3  setting1         128.1            121.0                 143.7   
            setting2         128.1            123.7                  44.4   
            setting3         128.1             47.1                 127.0   
            setting4         128.1            876.4                  50.3   
cologne3    setting1          63.9             50.7                 113.1   
            setting2          63.9            122.7                 103.4   
            setting3          63.9            111.1                  96.3   
            setting4          63.9            176.3                 297.9   
ingolstadt7 setting1          63.9             39.2                  45.8   
            setting2          63.9             41.5                  51.8   
            setting3          63.9             54.4                  69.9   
            setting4          63.9            333.0                 135.8   
hz4x4       setting1         349.2             58.7                  44.3   
            setting2         349.2             58.2                  50.2   
            setting3         349.2             61.9                  59.7   
            setting4         349.2             88.6                  73.4   

method                domain_adaptation    gat   ugat  jlgat  
network     setting                                           
tempe_1x1   setting1               17.1   18.1   28.4    NaN  
            setting2               20.7   17.8   22.2    NaN  
            setting3               25.9   33.2   25.1    NaN  
            setting4               89.4  124.6   83.5    NaN  
bullhead_1  setting1               16.3   16.2   17.3    NaN  
            setting2               14.7    7.8   12.4    NaN  
            setting3               19.9   15.8   22.2    NaN  
            setting4               28.7   30.9   20.2    NaN  
cologne1    setting1               84.8  116.6  114.8    NaN  
            setting2              110.2  114.4  121.8    NaN  
         

## Gap Bar Chart

In [9]:
import matplotlib.pyplot as plt


def plot_gap_bars(agent, df, networks, methods, settings, xticklabels, figsize=(30, 9)):
    sub = df[df['agent'] == agent]
    fig, axes = plt.subplots(2, 5, figsize=figsize, squeeze=False)
    x = np.arange(len(settings))
    width = 0.8 / len(methods)
    for ax, network in zip(axes.flat, networks):
        net_df = sub[sub['network'] == network]
        for i, method in enumerate(methods):
            vals = [
                net_df[(net_df['setting'] == s) & (net_df['method'] == method)]['gap_vs_cityflow'].mean()
                for s in settings
            ]
            ax.bar(x + (i - (len(methods) - 1) / 2) * width, vals, width * 0.9,
                   color=METHOD_COLORS[method], label=method)
        ax.axhline(0, color='0.4', linewidth=0.8)
        ax.set_title(str(network), fontsize=10)
        ax.set_xticks(x)
        ax.set_xticklabels(xticklabels, fontsize=7, rotation=90)
        ax.tick_params(labelsize=8)
        ax.spines[['top', 'right']].set_visible(False)
    for ax in axes.flat[len(networks):]:
        ax.set_visible(False)
    axes.flat[0].legend(loc='upper left', fontsize=8, frameon=False)
    fig.suptitle(f'{agent}: real-env ATT gap vs pretrained-cityflow baseline (lower is better)')
    fig.supylabel('ATT gap (s)')
    fig.tight_layout()
    plt.show()


for agent in AGENTS:
    plot_gap_bars(agent, results_df, NETWORK_ORDER, METHOD_ORDER, SETTING_ORDER, SETTING_ORDER)

## Real-Eval Travel-Time Traces

Replay curves per checkpoint; limited to two representative networks — edit `TRACE_NETWORKS` for more.

In [11]:
# Real-eval travel-time curves per checkpoint. direct_transfer (single zero-shot row)
# is drawn as a dashed reference line. 2x2 grid = the 4 transition settings.
TRACE_NETWORKS = ['tempe_1x1', 'tempe_16']


def real_trace(agent, network, method, setting):
    for log_path in reversed(candidate_dtls(agent, network, method, setting)):
        rows = real_rows(read_dtl(log_path))
        if len(rows) >= 2:
            return rows.sort_values('episode')
    return None


for agent in AGENTS:
    for network in TRACE_NETWORKS:
        fig, axes = plt.subplots(2, 2, figsize=(12, 7), squeeze=False)
        for ax, setting in zip(axes.flat, SETTING_ORDER):
            for method in METHOD_ORDER:
                if method in SINGLE_SHOT_METHODS:
                    dt = results_df[(results_df['agent'] == agent) & (results_df['network'] == network)
                                    & (results_df['setting'] == setting) & (results_df['method'] == method)]
                    if len(dt):
                        ax.axhline(float(dt['travel_time'].iloc[0]), color=METHOD_COLORS[method],
                                   linestyle='--', linewidth=1.0, label=method)
                    continue
                rows = real_trace(agent, network, method, setting)
                if rows is None:
                    continue
                ax.plot(rows['episode'], rows['travel_time'], color=METHOD_COLORS[method],
                        linewidth=1.2, label=method)
            ax.set_title(setting, fontsize=9)
            ax.tick_params(labelsize=7)
            ax.spines[['top', 'right']].set_visible(False)
        handles, labels = axes[0][0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper right', fontsize=9, frameon=False, ncols=4)
        fig.suptitle(f'{agent} / {network}: real-eval travel time per checkpoint', x=0.12, ha='left')
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        plt.show()